In [1]:
!pip install -q faster-whisper
!apt-get update -qq && apt-get install -y -qq fonts-liberation fonts-noto > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.2 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# Verify GPU is available (Colab should give you a T4)
import torch

print(f" GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print(" Warning: No GPU detected. Go to Runtime > Change runtime type > GPU")

# Check ffmpeg
import subprocess
try:
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("FFmpeg is ready")
except:
    !apt-get update -qq && apt-get install -y -qq ffmpeg
    print("FFmpeg installed")

 GPU Available: True
 GPU Name: Tesla T4
FFmpeg is ready


In [3]:
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import files
from faster_whisper import WhisperModel
import argparse
import os
import tempfile
import torch


In [4]:
def extract_audio(video_path: str, audio_output_path: str):
    """Extracts audio to 16kHz mono WAV (Whisper optimal format)"""
    print(f" Extracting audio from: {os.path.basename(video_path)}")
    command = [
        "ffmpeg",
        "-y",  # Overwrite without asking
        "-i", video_path,
        "-vn",  # No video
        "-acodec", "pcm_s16le",  # 16-bit PCM
        "-ar", "16000",  # 16kHz (Whisper requirement)
        "-ac", "1",  # Mono
        audio_output_path
    ]


    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Audio extracted")

In [5]:
def create_video_from_audio(audio_path: str, output_video_path: str, resolution="1080x1920"):
    """
    Creates a black background video from audio (for audiogram-style output).
    Essential for Colab since you can't burn subtitles into audio-only files.
    """
    print(f"Creating video from audio file...")

    # Get duration of audio
    probe_cmd = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", audio_path
    ]
    result = subprocess.run(probe_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    duration = float(result.stdout.strip())

    # Create black video with audio
    command = [
        "ffmpeg",
        "-y",
        "-f", "lavfi", "-i", f"color=c=black:s={resolution}:d={duration}",
        "-i", audio_path,
        "-shortest",  # Match duration to audio
        "-c:v", "libx264", "-preset", "fast",
        "-c:a", "aac", "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        output_video_path
    ]

    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Created video: {output_video_path}")


In [18]:
def transcribe_audio(audio_path: str, model_size: str = "small"):
    # Changed: "base" → "small" (much better accuracy, ~30% slower)
    # Added: word_timestamps=True for per-word timing
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    segments, info = model.transcribe(
        audio_path,
        beam_size=5,
        word_timestamps=True  # KEY: enables per-word timing
    )
    return list(segments), info

In [7]:
def format_timestamp(seconds: float) -> str:
    """Convert seconds to SRT format: HH:MM:SS,mmm"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

In [8]:
def generate_srt(segments, output_path: str):
    """Creates SRT file from segments"""
    # Safety check: if somehow segments is wrapped in another list
    if isinstance(segments, tuple):
        segments = segments[0]
    with open(output_path, "w", encoding="utf-8") as f:
        for i, segment in enumerate(segments, start=1):
            start_time = format_timestamp(segment.start)
            end_time = format_timestamp(segment.end)
            text = segment.text.strip()
            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{text}\n\n")
    print(f"SRT saved: {output_path}")


In [9]:
import re

def srt_to_ass_dynamic(srt_path: str, ass_path: str,
                        style: str = "stroke",
                        video_width: int = 1080,
                        video_height: int = 1920):
    """
    Converts SRT → ASS with per-caption font size scaling.
    Each caption gets its own override tag {\\fs} based on text length.
    Supports styles: "stroke", "yellow", "pill"
    """
    # Base style definitions
    style_defs = {
        "stroke": "Arial Black,72,&H00FFFFFF,&H000000FF,&H00000000,&H00000000,1,0,0,0,100,100,0,0,1,5,0,2,10,10,120,1",
        "yellow": "Arial Black,72,&H00FFFFFF,&H00FFE500,&H00000000,&H00000000,1,0,0,0,100,100,0,0,1,4,2,2,10,10,120,1",
        "pill":   "Arial Black,72,&H00FFFFFF,&H000000FF,&HAA000000,&H00000000,1,0,0,0,100,100,0,0,4,0,0,2,10,10,120,1",
    }
    chosen = style_defs.get(style, style_defs["stroke"])

    # ASS header
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {video_width}
PlayResY: {video_height}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{chosen}

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    def srt_time_to_ass(t):
        # "00:00:01,500" → "0:00:01.50"
        hh, mm, rest = t.split(":")
        ss, ms = rest.split(",")
        return f"{int(hh)}:{mm}:{ss}.{ms[:2]}"

    # Parse SRT blocks
    with open(srt_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    blocks = re.split(r"\n\s*\n", content)

    events = []
    for block in blocks:
        lines = block.strip().splitlines()
        if len(lines) < 3: continue
        if "-->" not in lines[1]: continue
        start_s, end_s = [x.strip() for x in lines[1].split("-->")]
        text = " ".join(lines[2:]).strip()
        if not text: continue

        # Dynamic font size per caption
        fs = get_safe_font_size(text, video_width=video_width)
        # Override tag: {\fs56\an2} = font size + bottom-center alignment
        ass_text = f"{{\\fs{fs}\\an2}}" + text.upper()

        start_a = srt_time_to_ass(start_s)
        end_a   = srt_time_to_ass(end_s)
        events.append(f"Dialogue: 0,{start_a},{end_a},Default,,0,0,0,,{ass_text}")

    with open(ass_path, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events))
    print(f"ASS file created: {ass_path} ({len(events)} captions)")

In [10]:
def burn_ass_subtitles(video_path: str, ass_path: str, output_path: str):
    """Burns .ass subtitles — supports per-caption font size overrides."""
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"ass='{ass_path}':fontsdir=/usr/share/fonts",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-c:a", "copy", "-movflags", "+faststart",
        output_path
    ]
    subprocess.run(cmd, check=True)
    print(f" Video with dynamic captions: {output_path}")

In [11]:
def get_safe_font_size(text: str, video_width: int = 1080,
                        max_size: int = 72, min_size: int = 36) -> int:
    """
    Scales font size so the longest word in 'text' always fits
    within ~85% of the video width (safe zone with margins).
    Arial Black: ~0.65px per char per pt at typical sizes.
    """
    safe_width = video_width * 0.82  # 82% = 9% margin each side
    longest = max(text.split(), key=len) if text.strip() else text
    chars = len(longest)
    # Empirical ratio for Arial Black: ~0.62 * font_size per char
    ideal = int(safe_width / (chars * 0.62))
    return max(min_size, min(max_size, ideal))

In [12]:
def make_word_level_srt(segments, output_path: str, words_per_flash: int = 2):
    """
    Uses actual word-level timestamps from Whisper (word_timestamps=True).
    Each flash = exactly N words, timed to when they were ACTUALLY spoken.
    Far more accurate than dividing segment time evenly.
    """
    idx = 1
    with open(output_path, "w", encoding="utf-8") as f:
        for segment in segments:
            words = segment.words  # requires word_timestamps=True
            if not words:
                continue
            # Chunk into groups of N words
            chunks = [words[i:i+words_per_flash] for i in range(0, len(words), words_per_flash)]
            for chunk in chunks:
                start = format_timestamp(chunk[0].start)
                end   = format_timestamp(chunk[-1].end)
                text  = " ".join(w.word.strip() for w in chunk)
                f.write(f"{idx}\n{start} --> {end}\n{text}\n\n")
                idx += 1
    print(f"Word-level SRT: {output_path} ({idx-1} flashes)")

In [13]:
print(" Upload a video file (mp4, mov, avi)...")
uploaded = files.upload()

# Store the filename
video_filename = list(uploaded.keys())[0]
print(f" Uploaded: {video_filename}")
print(f"File size: {os.path.getsize(video_filename) / (1024*1024):.2f} MB")

 Upload a video file (mp4, mov, avi)...


Saving test2.mp4 to test2.mp4
 Uploaded: test2.mp4
File size: 2.08 MB


In [14]:
INPUT_VIDEO = "/content/test2.mp4"  # Path to your uploaded video
MODEL_SIZE = "base"                      # Options: tiny, base, small, medium
BURN_SUBTITLES = True                    # Set to False if you only want the SRT file
OUTPUT_VIDEO_NAME = "my_videocaptionedd.mp4"

In [20]:
def run_pipeline(input_video, model_size="base", burn=True, output_name="output.mp4"):
    """
    Complete pipeline: Video → Audio → Text → Subtitles → Final Video
    """
    print(f" Starting pipeline for: {os.path.basename(input_video)}")

    # Setup paths
    input_path = Path(input_video)
    if not input_path.exists():
        raise FileNotFoundError(f" File not found: {input_video}")

    # Default SRT name (same as video but .srt)
    srt_path = str(input_path.with_suffix(".srt"))
    output_video = f"/content/{output_name}"

    # Temporary folder for audio extraction
    temp_dir = tempfile.mkdtemp()
    temp_audio = os.path.join(temp_dir, "audio.wav")

    try:
        # Step 1: Extract Audio from Video
        print("\nStep 1: Extracting audio...")
        extract_audio(str(input_path), temp_audio)

        # Step 2: AI Transcription (Whisper)
        print(f"\nStep 2: Transcribing with Whisper ({model_size})...")
        segments, info = transcribe_audio(temp_audio, model_size=model_size)
        print(f"   Found {len(segments)} text segments")

        # Step 3: Create SRT File
        print("\nStep 3: Generating SRT file...")
        generate_srt(segments, srt_path)
        print(f"   Saved: {srt_path}")
        # after your generate_srt(segments, srt_path)
        word_srt = str(Path(srt_path).with_name(Path(srt_path).stem + "_words.srt"))
        make_word_level_srt(segments, word_srt, words_per_flash=2)


        # Step 4: Burn Subtitles into Video (Optional)
        if burn:
            print(f"\n Step 4: Burning subtitles into video...")
            if burn:
              ass_path = word_srt.replace(".srt", ".ass")
              srt_to_ass_dynamic(
                  word_srt, ass_path,
                  style="pill",        # "stroke" | "yellow" | "pill"
                  video_width=1080,      # match your video resolution
                  video_height=1920     # 1080x1920 = standard TikTok/Reels
                  )
            burn_ass_subtitles(str(input_path), ass_path, output_video)
            # Show results
            size_mb = os.path.getsize(output_video) / (1024*1024)
            print(f"\n SUCCESS!")
            print(f"  Video: {output_video} ({size_mb:.1f} MB)")
            print(f"  Subtitles: {srt_path}")

            # Auto-download (optional)
            print("\n⬇ Downloading files...")
            files.download(output_video)
            files.download(srt_path)

            return output_video, srt_path, segments, info
        else:
            print("\n SRT file only generated (no video burning)")
            files.download(srt_path)
            return None, srt_path

    finally:
        # Cleanup temporary files
        if os.path.exists(temp_audio):
            os.remove(temp_audio)
        os.rmdir(temp_dir)
        print("\n Cleaned up temporary files")


In [21]:
final_video, subtitle_file, segments, info = run_pipeline(
    input_video=INPUT_VIDEO,
    model_size=MODEL_SIZE,
    burn=BURN_SUBTITLES,
    output_name=OUTPUT_VIDEO_NAME
)

 Starting pipeline for: test2.mp4

Step 1: Extracting audio...
 Extracting audio from: test2.mp4
Audio extracted

Step 2: Transcribing with Whisper (base)...
   Found 2 text segments

Step 3: Generating SRT file...
SRT saved: /content/test2.srt
   Saved: /content/test2.srt
Word-level SRT: /content/test2_words.srt (8 flashes)

 Step 4: Burning subtitles into video...
ASS file created: /content/test2_words.ass (8 captions)
 Video with dynamic captions: /content/my_videocaptionedd.mp4

 SUCCESS!
  Video: /content/my_videocaptionedd.mp4 (2.1 MB)
  Subtitles: /content/test2.srt

⬇ Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Cleaned up temporary files


In [22]:

import pickle
from pathlib import Path

# ── 1. Re-run (or reuse) the transcription ──────────────────
# If you already ran run_pipeline() above, segments + info are
# already in memory from transcribe_audio(). Re-call it here
# only if you skipped that step.

PKL_SAVE_PATH = "/content/transcription_result.pkl"   # ← change if you like

# Build the payload — info is a NamedTuple, so convert to dict for safety
payload = {
    "segments": segments,          # list of Whisper Segment objects
    "info": {                      # TranscriptionInfo fields
        "language":             info.language,
        "language_probability": info.language_probability,
        "duration":             info.duration,
        "duration_after_vad":   getattr(info, "duration_after_vad", None),
        "all_language_probs":   getattr(info, "all_language_probs", None),
    },
    "model_size": MODEL_SIZE,
    "source_video": INPUT_VIDEO,
}

with open(PKL_SAVE_PATH, "wb") as f:
    pickle.dump(payload, f)

pkl_size = Path(PKL_SAVE_PATH).stat().st_size / 1024
print(f" Saved pickle: {PKL_SAVE_PATH}  ({pkl_size:.1f} KB)")
print(f"   Segments : {len(segments)}")
print(f"   Language : {info.language} (p={info.language_probability:.2f})")
print(f"   Duration : {info.duration:.1f}s")

# ── 2. Auto-download the pkl to your machine ────────────────
from google.colab import files
files.download(PKL_SAVE_PATH)

 Saved pickle: /content/transcription_result.pkl  (3.4 KB)
   Segments : 2
   Language : en (p=0.98)
   Duration : 14.0s


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:



# ══════════════════════════════════════════════════════════════
# CELL: Load pkl back (use in a later session)
# ══════════════════════════════════════════════════════════════

# import pickle
#
# with open("/content/transcription_result.pkl", "rb") as f:
#     saved = pickle.load(f)
#
# segments  = saved["segments"]
# info_dict = saved["info"]
# print(f"Loaded {len(segments)} segments — language: {info_dict['language']}")
#
# # Regenerate SRT directly from the saved segments (no re-transcription needed)
# generate_srt(segments, "/content/reloaded.srt")
# make_word_level_srt(segments, "/content/reloaded_words.srt", words_per_flash=2)